# 30 · Routing: prefix-aware and KV-aware balancing
Level L5 · Part 5 · core · ~20 min · builds on lessons 29, 14

Time: about 20 min. This lesson starts **three simulated vLLM replicas** in Docker Compose (three small Python containers; the first start builds the image). The replicas are **SIMULATED**: they run no model, they only take time and report vLLM metric names. Without Docker, the table comes from the **recorded output** of a live run (`data/recorded/30.json`), labelled as such.

Ports: 8300 to 8302 (replicas r1 to r3, container port 8000).

## Goal
By the end you can route the same traffic three ways (round-robin, prefix-aware, token-budget), read a replica's prefix-cache counters, and explain the trade-off the numbers show: a router that keeps a prompt on one replica gets cache hits, and a router that counts tokens keeps replicas level.

## Why it matters
- **Without it:** round-robin takes turns. A 1,500-token policy prompt is cached on the replica that served it last, so the next request for that policy, sent elsewhere, prefills all 1,500 tokens again. Round-robin also counts requests: a 320-token answer holds about ten times the KV memory of a 32-token one, yet both count as one request, so one replica can fill up while another sits idle.
- **With it:** the router sends a request where its leading prompt blocks are already cached, or where the fewest tokens are in flight. Hits rise and KV memory stays level.
- **When you don't need it:** one replica, or prompts that never share a prefix. Routing then only adds a hop.

## The idea
A router in front of several model replicas picks one per request. A prefix is the start of a prompt. If two prompts begin with the same tokens, the server can reuse the KV cache it saved for them: that reuse is a **KV cache hit**. The cache works in 16-token blocks, from the first block up to the first difference. So the router should send a request where its leading blocks are already cached. KV memory is also a token budget, so it should count tokens in flight, not requests. Round-robin does neither. Think of a hotel desk that puts a tour group on the floor where its rooms are already made up, and sends the next group elsewhere once that floor is full. Neither rule is free: the prefix rule can pile load on one replica, and the token rule scatters prompts and loses the cache hits.

## Picture
```mermaid
flowchart LR
  C["Client: 120 requests"] --> P["Router (policy under test)"]
  P --> R1[r1]
  P --> R2[r2]
  P --> R3[r3]
  N["Rules, one per run: round-robin = next in turn; prefix-aware = most cached leading blocks; token-budget = fewest tokens in flight"] -.- P
```

## Step by step

### Step 1: the simulated replicas and their metric names
Each replica is a small FastAPI server. A request waits for one of four decode slots, sleeps for the time its uncached prompt and its answer would take, and returns text that says no model ran. It keeps a prefix cache of 16-token blocks, as lesson 14 explains. A word counts as one token. The names below are the ones vLLM exposes on `/metrics` (RESEARCH.md; the prefix-cache counters were checked on vLLM 0.32.0), so the router reads the same names a real engine would.

| Name on `/metrics` | Type | Meaning |
| --- | --- | --- |
| `vllm:num_requests_running` | gauge | requests in the batch |
| `vllm:num_requests_waiting` | gauge | requests waiting for a slot |
| `vllm:kv_cache_usage_perc` | gauge | KV memory in use, 0 to 1 |
| `vllm:prefix_cache_queries_total` | counter | prompt tokens queried against the cache |
| `vllm:prefix_cache_hits_total` | counter | prompt tokens found in the cache |
| `vllm:request_success_total` | counter | finished requests |

In [1]:
from pathlib import Path
Path("../apps/30").mkdir(parents=True, exist_ok=True); Path("../data/30").mkdir(parents=True, exist_ok=True)

The replica file is written next. All three replicas run this file; the compose file gives each one its name (`r1`, `r2`, `r3`). The `/sim/reset` route empties the cache between runs, so one run does not inherit another's hits. It exists only in this simulation.

In [2]:
%%writefile ../apps/30/mock_vllm.py
"""SIMULATED vLLM replica for lesson 30. It runs NO model: it takes time, keeps a prefix cache of
16-token blocks and exposes vLLM metric names on /metrics. One word counts as one token."""
import asyncio, hashlib, sys
from collections import OrderedDict
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import PlainTextResponse
from prometheus_client import CollectorRegistry, Counter, Gauge, generate_latest

NAME = sys.argv[1] if len(sys.argv) > 1 else "sim"
BLOCK, SLOTS, KV_CAPACITY, CACHE_BLOCKS = 16, 4, 12000, 200   # block size, decode slots, KV tokens, cached blocks
PREFILL_MS, DECODE_MS = 0.2, 5.0                              # simulated ms per uncached prompt token, per output token
REG = CollectorRegistry()
RUNNING = Gauge("vllm:num_requests_running", "Requests in the batch", registry=REG)
WAITING = Gauge("vllm:num_requests_waiting", "Requests waiting for a slot", registry=REG)
KV_USE = Gauge("vllm:kv_cache_usage_perc", "KV cache usage (0 to 1)", registry=REG)
# prometheus_client adds _total on the page: vllm:prefix_cache_queries_total
QUERIES = Counter("vllm:prefix_cache_queries", "Prompt tokens queried against the prefix cache", registry=REG)
HITS = Counter("vllm:prefix_cache_hits", "Prompt tokens found in the prefix cache", registry=REG)
SUCCESS = Counter("vllm:request_success", "Finished requests", registry=REG)
SLOT, CACHE, IN_FLIGHT = asyncio.Semaphore(SLOTS), OrderedDict(), [0]   # CACHE: block hash -> None, least recent first

def block_hashes(words):                                      # full blocks only, a hash chain as in lesson 14
    out, parent = [], b""
    for i in range(0, len(words) - BLOCK + 1, BLOCK):
        parent = hashlib.sha256(parent + " ".join(words[i:i + BLOCK]).encode()).digest()
        out.append(parent)
    return out

app = FastAPI()

@app.post("/v1/chat/completions")
async def chat(req: Request):
    body = await req.json()
    words = " ".join(m["content"] for m in body["messages"]).split()
    n_out, hashes = int(body.get("max_tokens", 32)), block_hashes(words)
    hit = 0
    while hit < len(hashes) and hashes[hit] in CACHE: hit += 1   # walk from block 0; stop at the first miss
    QUERIES.inc(len(words)); HITS.inc(hit * BLOCK)
    for h in hashes:                                          # store the blocks, evicting the least recently used
        CACHE.pop(h, None); CACHE[h] = None
        if len(CACHE) > CACHE_BLOCKS: CACHE.popitem(last=False)
    need = len(words) + n_out
    IN_FLIGHT[0] += need; WAITING.inc()
    async with SLOT:                                          # waits here while all slots are busy
        WAITING.dec(); RUNNING.inc()
        await asyncio.sleep(((len(words) - hit * BLOCK) * PREFILL_MS + n_out * DECODE_MS) / 1000)
        RUNNING.dec(); IN_FLIGHT[0] -= need
    SUCCESS.inc()
    return {"id": f"sim-{NAME}", "object": "chat.completion", "model": body.get("model"), "replica": NAME,
            "choices": [{"index": 0, "finish_reason": "length", "message": {"role": "assistant", "content": f"[SIMULATED replica {NAME}: no model ran]"}}],
            "usage": {"prompt_tokens": len(words), "completion_tokens": n_out, "total_tokens": need,
                      "prompt_tokens_details": {"cached_tokens": hit * BLOCK}}}

@app.get("/metrics")
def metrics():
    KV_USE.set(min(1.0, (IN_FLIGHT[0] + len(CACHE) * BLOCK) / KV_CAPACITY))
    return PlainTextResponse(generate_latest(REG).decode(), media_type="text/plain")

@app.post("/sim/reset")                                       # simulation only: empty the cache between runs
def reset():
    CACHE.clear(); return {"cache_blocks": 0}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

Overwriting ../apps/30/mock_vllm.py


### Step 2: start the three replicas
Check Docker first: `docker info` should succeed. The compose file `infra/compose/routing-3replica.yml` builds the image on a pinned Python base and starts r1 to r3. You do not need to edit it. If port 8300 is taken by another program, stop that program, or change the host ports in the compose file and in `REPLICAS`. **To run this live:** set `TUT_ENABLE=docker` before you start Jupyter. Run all the `needs-docker` cells or none. If Docker is not running, those cells are skipped and every table row is labelled "(recorded)".

The imports and helpers come first. The project name only keeps this lab's containers apart from other courses.

In [3]:
import asyncio, atexit, hashlib, itertools, json, random, subprocess, time
import httpx
from prometheus_client.parser import text_string_to_metric_families

PROJECT, COMPOSE = "tut-kv-forge-15h-30", "../infra/compose/routing-3replica.yml"
REPLICAS = {"r1": "http://127.0.0.1:8300", "r2": "http://127.0.0.1:8301", "r3": "http://127.0.0.1:8302"}
POLICIES = ["round-robin", "prefix-aware", "token-budget"]
RESULTS, RECORDED, LIVE = {}, set(), False

def recorded():
    """The recorded live run (data/recorded/30.json): used for any row or page that was not run live."""
    return json.loads(Path("../data/recorded/30.json").read_text())

def compose(*args):
    r = subprocess.run(["docker", "compose", "-p", PROJECT, "-f", COMPOSE, *args], capture_output=True, text=True)
    if r.returncode: raise RuntimeError(r.stderr[-800:])

def sample_values(text):                                      # the Prometheus client parses a page: {sample name: value}
    return {s.name: s.value for f in text_string_to_metric_families(text) for s in f.samples}

Start the containers. The cell waits for every replica to answer `/metrics`, then sets `LIVE`, so the later live cells run.

In [4]:
def start_cluster(limit=180):
    global LIVE
    compose("up", "-d", "--build")
    for url in REPLICAS.values():
        for _ in range(limit):
            try:
                if httpx.get(url + "/metrics", timeout=2).status_code == 200: break
            except httpx.TransportError: pass
            time.sleep(1)
        else: raise TimeoutError(f"no answer from {url}")
    LIVE = True
    atexit.register(lambda: LIVE and compose("down", "-v"))
start_cluster()

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


### Step 3: the routing policies (hand-written, on purpose)
Three rules, one per run. **round-robin** takes the next replica in turn. **token-budget** takes the replica with the fewest estimated tokens in flight. **prefix-aware** picks the replica that has seen the most leading blocks of this prompt. If none has, or if it already has `CAP` requests in flight, it picks the replica with the fewest tokens.

The routing code is hand-written because it *is* the topic of this lesson. LiteLLM from lesson 29 balances on request counts, latency, cost or usage, not on prompt blocks (LiteLLM routing docs, checked 2026-10-23). The vLLM production-stack router does round-robin and session routing; its prefix-aware routing is work in progress (RESEARCH.md, section 8.3). The Gateway API Inference Extension's endpoint picker reads server metrics, including prefix-cache status, and llm-d is the suggested production picker (RESEARCH.md, section 8.3).

**CAP** is the in-flight limit for the prefix rule. With 24 requests in flight over 3 replicas, each replica averages 8, and each replica has 4 decode slots. A cap below that average would likely skip the cached replica often (not measured here), so the cap is 10.

In [5]:
BLOCK, CAP = 16, 10                                           # tokens per cache block; in-flight limit for the prefix rule

def block_hashes(words):
    """Hash chain over full 16-token blocks (lesson 14): each hash covers its block and the one before it."""
    out, parent = [], b""
    for i in range(0, len(words) - BLOCK + 1, BLOCK):
        parent = hashlib.sha256(parent + " ".join(words[i:i + BLOCK]).encode()).digest()
        out.append(parent)
    return out

print(len(block_hashes([f"w{i}" for i in range(1500)])), "full 16-token blocks in a 1,500-token prompt")

93 full 16-token blocks in a 1,500-token prompt


The router is a small class. `pick` applies the policy to one request and records the choice. `done` releases its tokens when the answer arrives.

In [6]:
class Router:
    """One policy per run. It keeps tokens in flight, requests in flight and the blocks it sent to each replica."""
    def __init__(self, policy):
        self.policy, self.turn = policy, itertools.cycle(REPLICAS)
        self.tokens = dict.fromkeys(REPLICAS, 0)              # estimated tokens in flight: prompt + max_tokens
        self.busy = dict.fromkeys(REPLICAS, 0)                # requests in flight
        self.seen = {r: set() for r in REPLICAS}              # block hashes this router sent to each replica
    def pick(self, words, n_out):
        lightest = min(REPLICAS, key=self.tokens.get)
        if self.policy == "round-robin":
            r = next(self.turn)
        elif self.policy == "token-budget":
            r = lightest
        else:                                                 # prefix-aware: most leading blocks seen there, unless busy
            hs = block_hashes(words)
            def match(x): return len(list(itertools.takewhile(self.seen[x].__contains__, hs)))
            best = max(REPLICAS, key=match)
            r = best if match(best) and self.busy[best] < CAP else lightest
        self.seen[r].update(block_hashes(words))
        self.tokens[r] += len(words) + n_out; self.busy[r] += 1
        return r
    def done(self, r, words, n_out):
        self.tokens[r] -= len(words) + n_out; self.busy[r] -= 1

print("routing policies:", ", ".join(POLICIES))

routing policies: round-robin, prefix-aware, token-budget


### Step 4: the same 120 requests, three policies
The workload is fixed (seed 30): 120 requests, each a 1,500-token policy prompt from one of three families plus a 20 to 60 token question. 80% ask for a 32-token answer and 20% for 320 tokens. The client keeps 24 requests in flight.

In [7]:
rng = random.Random(30)
FAMILY_WORDS = {f: [f"{f}-clause-{i}" for i in range(1500)] for f in ("refunds", "shipping", "accounts")}
def request():
    words = FAMILY_WORDS[rng.choice(list(FAMILY_WORDS))] + [f"q{rng.randrange(500)}" for _ in range(rng.randint(20, 60))]
    return words, 32 if rng.random() < 0.8 else 320
REQS = [request() for _ in range(120)]
print(len(REQS), "requests,", sum(len(w) for w, _ in REQS), "prompt tokens,", sum(n for _, n in REQS), "answer tokens")

120 requests, 184728 prompt tokens, 10464 answer tokens


The client sends the requests. The router picks a replica as each request is sent.

In [8]:
async def run(policy, in_flight=24):
    router, gate, lat, errors = Router(policy), asyncio.Semaphore(in_flight), [], 0
    async with httpx.AsyncClient(timeout=120) as client:
        async def one(words, n_out):
            nonlocal errors
            async with gate:
                r = router.pick(words, n_out)
                t0 = time.perf_counter()
                body = {"model": "brightside-qwen", "max_tokens": n_out, "messages": [{"role": "user", "content": " ".join(words)}]}
                try:
                    resp = await client.post(REPLICAS[r] + "/v1/chat/completions", json=body)
                    resp.raise_for_status(); lat.append(time.perf_counter() - t0)
                except httpx.HTTPError: errors += 1
                finally: router.done(r, words, n_out)
        await asyncio.gather(*(one(w, n) for w, n in REQS))
    return lat, errors

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


`measure` empties the caches, reads each replica's counters before and after the run, and stores one row. The hit rate is replica-side: hit tokens over queried tokens.

In [9]:
def replica_totals():
    out = {}
    for name, url in REPLICAS.items():
        s = sample_values(httpx.get(url + "/metrics", timeout=5).text)
        out[name] = {k: s.get(f"vllm:{k}_total", 0.0) for k in ("prefix_cache_queries", "prefix_cache_hits", "request_success")}
    return out

async def measure(policy):
    for url in REPLICAS.values(): httpx.post(url + "/sim/reset", timeout=5)     # no hits carried over from another run
    before = replica_totals()
    lat, errors = await run(policy)
    after = replica_totals()
    queries = sum(after[r]["prefix_cache_queries"] - before[r]["prefix_cache_queries"] for r in REPLICAS)
    hits = sum(after[r]["prefix_cache_hits"] - before[r]["prefix_cache_hits"] for r in REPLICAS)
    served = {r: int(after[r]["request_success"] - before[r]["request_success"]) for r in REPLICAS}
    RESULTS[policy] = {"ok": len(lat), "errors": errors, "hit_rate": round(hits / queries, 3), "served": served,
                       "p95_s": round(sorted(lat)[int(0.95 * len(lat)) - 1], 2)}

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Run the three policies in turn. Without Docker this cell does nothing, and the table comes from the recording.

In [10]:
if LIVE:
    for p in POLICIES:
        await measure(p)

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


### Step 5: read the table
The table is **simulated**: the replicas are simulated, and the p95 is simulated time, not measured vLLM latency. Every row has the same 120 requests and the same three replicas. `r1` to `r3` count what each replica finished, from its own `vllm:request_success_total`. A row that was not run live is filled from the recording and marked "(recorded)".

In [11]:
def show():
    for k in POLICIES:
        if k not in RESULTS:
            RESULTS[k] = recorded()["results"][k]; RECORDED.add(k)
    print(f"{'policy':<13} {'ok':>4} {'err':>4} {'prefix hit':>10}  {'r1':>4} {'r2':>4} {'r3':>4}  {'p95 s':>6}")
    for k in POLICIES:
        r, s = RESULTS[k], RESULTS[k]["served"]
        tag = "  (recorded)" if k in RECORDED else ""
        print(f"{k:<13} {r['ok']:>4} {r['errors']:>4} {r['hit_rate']:>10.0%}  {s['r1']:>4} {s['r2']:>4} {s['r3']:>4}  {r['p95_s']:>6}{tag}")

show()

policy          ok  err prefix hit    r1   r2   r3   p95 s
round-robin    120    0        60%    40   40   40    2.42  (recorded)
prefix-aware   120    0        84%    54   34   32    2.09  (recorded)
token-budget   120    0        58%    35   48   37    2.37  (recorded)


**Read it (simulated).** Round-robin spreads requests exactly evenly (40, 40, 40) and gets a 60% prefix hit rate. Its cache fills by repetition: 120 requests over 3 families means each replica sees each family again after a few requests, so it hits often. Token-budget balances tokens in flight, not requests. Its counts are uneven (35, 48, 37), and its hit rate (58%) is about round-robin's, probably because each family lands on all three replicas. Prefix-aware reaches 84% and has the lowest p95 (2.09 s). Its misses are probably the first request of each family on each replica, the requests that overflow the cap to the lightest replica, and the LRU limit: 3 families × 93 blocks = 279 blocks, more than one replica's 200-block cache holds. Its counts are the least even (54, 34, 32), probably because the cap and the family mix sent more requests to r1 in this seeded run.

Now read r1's `/metrics` page after the runs. Live, it is the real page; without Docker, it is the recorded page. These are the names from Step 1, with the values this replica counted.

In [12]:
text = httpx.get(REPLICAS["r1"] + "/metrics", timeout=5).text if LIVE else recorded()["metrics_text"]
print("source:", "this live run" if LIVE else "recorded output")
print("\n".join(l for l in text.splitlines() if l.startswith("vllm:") and "_created" not in l))

source: recorded output
vllm:num_requests_running 0.0
vllm:num_requests_waiting 0.0
vllm:kv_cache_usage_perc 0.26666666666666666
vllm:prefix_cache_queries_total 198559.0
vllm:prefix_cache_hits_total 139872.0
vllm:request_success_total 129.0


## At work
- **Meta / PyTorch (2025-09-12)**: sticky routing with load balancing reached a 40 to 50% prefix-cache hit rate at 90% HBM use, in a disaggregated stack. Hit rate and balance pull against each other, as this table shows. → lesson 33 (dashboards and reliability).
- **Cloudflare (2026-05)** tracks estimated in-flight prefill and decode tokens per endpoint, not requests; session affinity raised its peak input cache hit ratio from 60% to 80% (vendor-reported). → lesson 35 for changing routing safely.
- **Anthropic postmortem (2025-09-17)** found that a routing change sent short requests to servers built for a long context window, and that sticky routing kept follow-ups on the wrong servers. → lesson 35 (rollouts with eval gates).
- **Go deeper:** the Gateway API Inference Extension's endpoint picker and llm-d route on prefix-cache status in Kubernetes (RESEARCH.md, section 8.3; not taught here). llm-d's gain is vendor-reported.

## Common mistakes
**1. Reading a counter by the name the Python library gives it.** The counter is named `vllm:prefix_cache_hits` in the code, but the page shows `vllm:prefix_cache_hits_total`. A lookup with the short name fails. This cell reads the recorded page, so it runs without Docker. **This cell fails on purpose.**

In [13]:
hits = sample_values(recorded()["metrics_text"])["vllm:prefix_cache_hits"]
print(hits)

KeyError: 'vllm:prefix_cache_hits'

**The fix:** use the sample name as it appears on the page, with `_total`: `sample_values(text)["vllm:prefix_cache_hits_total"]`. Read the names from the page first, as the last step does.

**2. Comparing runs without emptying the cache.** The second policy would start with the first one's blocks, and its hit rate would be too high. `measure` calls the mock's `/sim/reset` route for that reason. A real engine has no such route in this lab, so restart it, or wait for a steady state, before you compare.

## Production checklist
- Route on the metric names your engine version exposes, and check them after every upgrade: vLLM has renamed gauges across releases (RESEARCH.md, section 7, row 12).
- Count tokens in flight (prompt plus `max_tokens`), not requests. One 320-token answer is not one short answer.
- Cap a busy replica, with the cap above the average load per replica (Step 3). A cap below that average would likely skip the cached replica often (not measured here).
- Read the hit rate and the spread of replica counts together. A hit rate that rises while one replica idles is not a win.
- A router with its own index needs at least two instances for availability, and each instance learns the cache only from its own traffic.

## Try it
From `RESULTS`, for each policy compute the spread between the busiest and the idlest replica (most minus fewest requests served), and name the policy with the best prefix hit rate and the one with the most even counts.

Run the Step 5 cell first: without Docker it fills `RESULTS` from the recording, and this exercise reads `RESULTS`.

In [14]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [15]:
for k in POLICIES:
    s = RESULTS[k]["served"]
    print(f"{k:<13} spread {max(s.values()) - min(s.values()):>3} requests | hit rate {RESULTS[k]['hit_rate']:.0%}")
best = max(POLICIES, key=lambda k: RESULTS[k]["hit_rate"])
even = min(POLICIES, key=lambda k: max(RESULTS[k]["served"].values()) - min(RESULTS[k]["served"].values()))
print("best prefix hit rate:", best, "| most even replica counts:", even)

round-robin   spread   0 requests | hit rate 60%
prefix-aware  spread  22 requests | hit rate 84%
token-budget  spread  13 requests | hit rate 58%
best prefix hit rate: prefix-aware | most even replica counts: round-robin


## Recap
- Round-robin takes turns and ignores the prompt cache and the token budget, so each family is prefilled again on replicas that have not cached it.
- Prefix-aware routing sends a request where its leading blocks are cached, unless that replica is busy. It raises hits and can unbalance load.
- Token-budget routing counts tokens in flight. It keeps replicas level and can lose cache hits.
- Read `vllm:prefix_cache_hits_total` and `vllm:prefix_cache_queries_total` by their full names, and read the hit rate next to the spread.
- **Interview:** "How do you route requests across replicas that cache prompts?" Strong answer: route on cache affinity and tokens in flight together, cap the hot replica, and measure the hit rate and the spread on your own traffic.

Save the live run as a run-time file (not committed) and stop the containers. Without Docker, this cell does nothing.

In [16]:
if LIVE:
    page = httpx.get(REPLICAS["r1"] + "/metrics", timeout=5).text          # r1's page after the runs
    Path("../data/30/results.json").write_text(json.dumps({"results": RESULTS, "metrics_text": page}, indent=1))
    print("saved the live run to data/30/results.json (run-time file, not committed)")
    compose("down", "-v"); LIVE = False

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.
